# Out-of-sample phenotype estimation

This notebook estimates participant coordinates in folds of participants that are excluded from model fitting. Preprocessing parameters, model components, covariance estimates, and PC axes are learned only from each training fold.

## Notebook flow

1. Create participant-level cross-fitting folds.
2. Fit and freeze the phenotype model within each training fold.
3. Estimate two split-half scores for each held-out participant.
4. Repeat the procedure for three-day and complementary monitoring windows.
5. Summarize pooled, fold-specific, and window-specific agreement.

## 1. Setup

Resolve project paths, import the cross-fitting utilities, and load the common prepared meal-level dataset.

In [1]:
from pathlib import Path
import sys

# Find the project when launched from its root, code/, or a notebook subfolder.
for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    CODE_DIR = PROJECT_ROOT / "code"
    if (CODE_DIR / "data_paths.py").is_file():
        break
else:
    raise FileNotFoundError("Open this notebook from within the project directory.")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Shared defaults; override individual paths here if needed.
from data_paths import DATA_DIR, METADATA_PATH, MEAL_DATA_PATH, CGM_METRICS_PATH
FIGURES_DIR = PROJECT_ROOT / "Figures"
RESULTS_DIR = PROJECT_ROOT / "Results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "OOS_PC_estimation").mkdir(parents=True, exist_ok=True)


In [2]:
import logging
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import xgboost as xgb
from mmer import MixedEffectEstimator
from sklearn.model_selection import BaseCrossValidator, GroupKFold
from tqdm import tqdm

In [3]:
from utils import (
    DATE_COL,
    OUTCOMES,
    RANDOM_SLOPES,
    SUBJECT_COL,
    build_design_mats,
    derive_pc_basis,
    fit_training_fold,
    icc_split_half,
    infer_blups_from_frozen_model,
    load_and_prepare_data,
    make_subject_folds,
    make_xgb,
    project_blups_to_pcs,
    split_half_within_subject,
    transform_validation_data,
)

In [4]:
meta_data, data, id_to_subject_key, _= load_and_prepare_data(
    METADATA_PATH,
    MEAL_DATA_PATH,
)

print(f"Prepared {len(data):,} meals from {data['subject_key'].nunique():,} participants.")


data shape : (54987, 133)
Prepared 54,987 meals from 992 participants.


## 2. Cross-fitted validation design

The same fold-specific workflow is used for both validation analyses below.

### Cross-fitted design

1. Split participants into 80% training and 20% held-out folds.
2. Fit preprocessing and MMER only on training participants.
3. Freeze scalers, the fixed-effect XGBoost model, covariance components, and PC basis.
4. Split held-out participants' meals into independent subsets.
5. Transform and score both subsets with the frozen training-fold model.
6. Compare held-out participant PC estimates without refitting the phenotype model.

## 3. Held-out split-half analysis

Prepare the eligible analysis sample, define the fold-level fitting and scoring functions, and run odd/even and chronological splits. The displayed score tables, diagnostics, and ICC summaries are calculated from held-out participants only.

In [5]:
import os
import numpy as np
import pandas as pd

from joblib import Parallel, delayed
from sklearn.model_selection import GroupKFold
from scipy.stats import pearsonr, spearmanr

import xgboost as xgb
from mmer import MixedEffectEstimator

In [6]:
N_FOLDS = 5

# Number of folds fitted simultaneously.
# With 4 threads inside each XGBoost/MMER model, start with 2 outer jobs.
N_JOBS_OUTER = 4
N_JOBS_MODEL = 8

RANDOM_SLOPES = [
    "carb_eaten",
    "fat_eaten",
    "protein_eaten",
    "fiber_eaten",
]

OUTCOMES = [
    "max_glucose",
    "peak_duration",
    "end_glucose",
    "positive_iAUC",
]

SUBJECT_COL = "subject_key"
DATE_COL = "eaten_date"

MIN_TOTAL_MEALS = 1
MIN_MEALS_PER_HALF = 1

In [7]:
def prepare_analysis_data(df_food):

    data[DATE_COL] = pd.to_datetime(data[DATE_COL])

    # Keep participants with enough total meals for reliable two-half inference.
    meal_counts = data.groupby(SUBJECT_COL).size()
    eligible_subjects = meal_counts[
        meal_counts >= MIN_TOTAL_MEALS
    ].index

    data = data[
        data[SUBJECT_COL].isin(eligible_subjects)
    ].copy()

    data = (
        data
        .sort_values([SUBJECT_COL, DATE_COL])
        .reset_index(drop=True)
    )

    return data

In [8]:

print("Meals:", len(data))
print("Participants:", data[SUBJECT_COL].nunique())

Meals: 54987
Participants: 992


In [9]:
def process_one_fold(
    fold_id,
    train_indices,
    test_indices,
    data,
    split_mode="even_odd",
    n_jobs_model=4,
    min_meals_per_half=8,
):
    """
    Fit one fold and produce two independently inferred PC scores
    for every eligible held-out participant.
    """
    train_subjects = data.iloc[train_indices][SUBJECT_COL].unique()
    test_subjects = data.iloc[test_indices][SUBJECT_COL].unique()

    # GroupKFold indexes participants; select meal rows by participant identifier.
    train_data = data[
        data[SUBJECT_COL].isin(train_subjects)
    ].copy()

    test_data = data[
        data[SUBJECT_COL].isin(test_subjects)
    ].copy()

    overlap = set(train_subjects).intersection(test_subjects)

    if overlap:
        raise RuntimeError(
            f"Participant leakage detected in fold {fold_id}."
        )

    print(
        f"Fold {fold_id}: "
        f"{len(train_subjects)} training participants, "
        f"{len(test_subjects)} held-out participants"
    )

    # 1. Fit using training participants only.
    fitted_result, fold_scalers, train_info = fit_training_fold(
        train_data=train_data,
        n_jobs_model=n_jobs_model,
    )

    # 2. Define PCs from training covariance only.
    eigenvalues, eigenvectors = derive_pc_basis(
        fitted_result
    )

    # 3. Split held-out participants.
    half_A, half_B, span = split_half_within_subject(
        test_data,
        split_mode=split_mode,
        subject_col=SUBJECT_COL,
        date_col=DATE_COL,
        min_meals_per_half=min_meals_per_half,
    )

    # 4. Infer two BLUP estimates using the same frozen model.
    blup_A = infer_blups_from_frozen_model(
        fitted_result=fitted_result,
        validation_data=half_A,
        fold_scalers=fold_scalers,
    )

    blup_B = infer_blups_from_frozen_model(
        fitted_result=fitted_result,
        validation_data=half_B,
        fold_scalers=fold_scalers,
    )

    common_participants = (
        blup_A.index
        .intersection(blup_B.index)
    )

    blup_A = blup_A.loc[common_participants]
    blup_B = blup_B.loc[common_participants]

    # 5. Project both BLUP estimates onto identical fold-specific PCs.
    scores_A = project_blups_to_pcs(
        blups=blup_A,
        eigenvalues=eigenvalues,
        eigenvectors=eigenvectors,
        n_components=2,
        standardize=True,
    )

    scores_B = project_blups_to_pcs(
        blups=blup_B,
        eigenvalues=eigenvalues,
        eigenvectors=eigenvectors,
        n_components=2,
        standardize=True,
    )

    fold_scores = (
        scores_A.add_suffix("_A")
        .join(scores_B.add_suffix("_B"))
        .reset_index()
    )

    participant_column = (
        blup_A.index.name
        if blup_A.index.name is not None
        else "index"
    )

    fold_scores = fold_scores.rename(
        columns={participant_column: SUBJECT_COL}
    )

    fold_scores["fold"] = fold_id
    fold_scores["split_mode"] = split_mode

    # Attach meal counts.
    count_table = (
        span.pivot(
            index=SUBJECT_COL,
            columns="half",
            values="n_meals",
        )
        .rename(
            columns={
                "A": "n_meals_A",
                "B": "n_meals_B",
            }
        )
        .reset_index()
    )

    fold_scores = fold_scores.merge(
        count_table,
        on=SUBJECT_COL,
        how="left",
    )

    fold_diagnostics = {
        "fold": fold_id,
        "split_mode": split_mode,
        "n_train_subjects": len(train_subjects),
        "n_test_subjects": len(test_subjects),
        "n_scored_subjects": len(fold_scores),
        "n_train_meals": len(train_data),
        "n_test_meals": len(test_data),
        "converged": fitted_result.is_converged,
        "log_likelihood": fitted_result.best_log_likelihood,
        "eigenvalue_1": eigenvalues[0],
        "eigenvalue_2": eigenvalues[1],
        "pc1_variance_fraction": (
            eigenvalues[0] / eigenvalues.sum()
        ),
        "pc2_variance_fraction": (
            eigenvalues[1] / eigenvalues.sum()
        ),
    }

    loading_table = pd.DataFrame({
        "coordinate": blup_A.columns,
        "PC1": eigenvectors[:, 0],
        "PC2": eigenvectors[:, 1],
        "fold": fold_id,
        "split_mode": split_mode,
    })

    return {
        "scores": fold_scores,
        "diagnostics": fold_diagnostics,
        "loadings": loading_table,
    }

In [10]:
def process_subject_fold(
    fold_id,
    train_subjects,
    test_subjects,
    data,
    split_mode="even_odd",
    n_jobs_model=4,
    min_meals_per_half=5,
):
    train_data = data[
        data[SUBJECT_COL].isin(train_subjects)
    ].copy()

    test_data = data[
        data[SUBJECT_COL].isin(test_subjects)
    ].copy()

    if set(train_subjects).intersection(test_subjects):
        raise RuntimeError("Training/test participant leakage.")

    fitted_result, fold_scalers, train_info = fit_training_fold(
        train_data,
        n_jobs_model=n_jobs_model,
    )

    eigenvalues, eigenvectors = derive_pc_basis(
        fitted_result
    )

    half_A, half_B, span = split_half_within_subject(
        test_data,
        split_mode=split_mode,
        min_meals_per_half=min_meals_per_half,
    )

    blup_A = infer_blups_from_frozen_model(
        fitted_result,
        half_A,
        fold_scalers,
    )

    blup_B = infer_blups_from_frozen_model(
        fitted_result,
        half_B,
        fold_scalers,
    )

    common = blup_A.index.intersection(blup_B.index)
    blup_A = blup_A.loc[common]
    blup_B = blup_B.loc[common]

    scores_A = project_blups_to_pcs(
        blup_A,
        eigenvalues,
        eigenvectors,
        n_components=2,
        standardize=True,
    )

    scores_B = project_blups_to_pcs(
        blup_B,
        eigenvalues,
        eigenvectors,
        n_components=2,
        standardize=True,
    )

    scores = (
        scores_A.add_suffix("_A")
        .join(scores_B.add_suffix("_B"))
        .reset_index()
    )

    index_name = blup_A.index.name or "index"

    scores = scores.rename(
        columns={index_name: SUBJECT_COL}
    )

    scores["fold"] = fold_id
    scores["split_mode"] = split_mode

    counts = (
        span.pivot(
            index=SUBJECT_COL,
            columns="half",
            values="n_meals",
        )
        .rename(
            columns={
                "A": "n_meals_A",
                "B": "n_meals_B",
            }
        )
        .reset_index()
    )

    scores = scores.merge(
        counts,
        on=SUBJECT_COL,
        how="left",
    )

    diagnostics = {
        "fold": fold_id,
        "split_mode": split_mode,
        "n_train_subjects": len(train_subjects),
        "n_test_subjects": len(test_subjects),
        "n_scored_subjects": len(scores),
        "converged": fitted_result.is_converged,
        "log_likelihood": fitted_result.best_log_likelihood,
        "pc1_variance_fraction": eigenvalues[0] / eigenvalues.sum(),
        "pc2_variance_fraction": eigenvalues[1] / eigenvalues.sum(),
    }

    loadings = pd.DataFrame({
        "coordinate": blup_A.columns,
        "PC1": eigenvectors[:, 0],
        "PC2": eigenvectors[:, 1],
        "fold": fold_id,
        "split_mode": split_mode,
    })

    return {
        "scores": scores,
        "diagnostics": diagnostics,
        "loadings": loadings,
    }

In [11]:
def run_cross_fitted_analysis(
    data,
    split_mode="even_odd",
    n_splits=5,
    n_jobs_outer=2,
    n_jobs_model=4,
    min_meals_per_half=5,
):
    folds = make_subject_folds(
        data,
        n_splits=n_splits,
    )

    results = Parallel(
        n_jobs=n_jobs_outer,
        backend="loky",
        verbose=10,
    )(
        delayed(process_subject_fold)(
            fold_id=fold["fold_id"],
            train_subjects=fold["train_subjects"],
            test_subjects=fold["test_subjects"],
            data=data,
            split_mode=split_mode,
            n_jobs_model=n_jobs_model,
            min_meals_per_half=min_meals_per_half,
        )
        for fold in folds
    )

    score_table = pd.concat(
        [result["scores"] for result in results],
        ignore_index=True,
    )

    diagnostic_table = pd.DataFrame(
        [result["diagnostics"] for result in results]
    )

    loading_table = pd.concat(
        [result["loadings"] for result in results],
        ignore_index=True,
    )

    return {
        "scores": score_table,
        "diagnostics": diagnostic_table,
        "loadings": loading_table,
    }

In [ ]:
odd_even_result = run_cross_fitted_analysis(
    data=data,
    split_mode="even_odd",
    n_splits=N_FOLDS,
    n_jobs_outer=4,
    n_jobs_model=8,
    min_meals_per_half=MIN_MEALS_PER_HALF, #MIN_MEALS_PER_HALF = 1
)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
Finished: no further improvement!:  40%|████      | 20/50 05:49         
Finished: no further improvement!:  42%|████▏     | 21/50 06:01         
[Parallel(n_jobs=4)]: Done   2 out of   5 | elapsed:  6.5min remaining:  9.7min
Finished: no further improvement!:  62%|██████▏   | 31/50 09:07         
[Parallel(n_jobs=4)]: Done   3 out of   5 | elapsed:  9.7min remaining:  6.4min
Finished: no further improvement!:  62%|██████▏   | 31/50 09:38         
Running MMER Framework | Fitting Model ...:  44%|████▍     | 22/50 05:18

In [ ]:
odd_even_result["scores"]

In [ ]:
odd_even_result["diagnostics"]

In [12]:
odd_even_result["diagnostics"]

,fold,split_mode,n_train_subjects,n_test_subjects,n_scored_subjects,converged,log_likelihood,pc1_variance_fraction,pc2_variance_fraction
0,1,even_odd,793,199,199,True,-88429.275972,0.665357,0.130563
1,2,even_odd,793,199,199,True,-89583.285474,0.682171,0.120676
2,3,even_odd,794,198,198,True,-89469.668162,0.668710,0.115054
3,4,even_odd,794,198,198,True,-88724.091488,0.700232,0.110309
4,5,even_odd,794,198,198,True,-90223.928526,0.672933,0.115917


In [13]:
print("odd/even split estimations :")
summary, full = icc_split_half(odd_even_result["scores"])
print(summary.to_string(index=False))

odd/even split estimations :
  fold  pc   n  ICC(A,1)  ICC(A,1) CI  ICC(C,1)  ICC(C,1) CI  pval    r         r CI  rho       rho CI
     1 PC1 199      0.84 [0.79, 0.87]      0.84 [0.79, 0.87]   0.0 0.84 [0.79, 0.87] 0.81 [0.75, 0.85]
     1 PC2 199      0.48 [0.36, 0.58]      0.48 [0.36, 0.58]   0.0 0.48 [0.36, 0.58] 0.45 [0.34, 0.56]
     2 PC1 199      0.81 [0.75, 0.85]      0.81 [0.75, 0.85]   0.0 0.81 [0.75, 0.85] 0.82 [0.77, 0.86]
     2 PC2 199      0.58 [0.48, 0.67]      0.58 [0.48, 0.67]   0.0 0.59 [0.49, 0.67] 0.48 [0.36, 0.58]
     3 PC1 198      0.83 [0.78, 0.87]      0.83 [0.78, 0.87]   0.0 0.83 [0.78, 0.87] 0.78 [0.72, 0.83]
     3 PC2 198      0.49 [0.37, 0.59]      0.49 [0.37, 0.58]   0.0 0.49 [0.37, 0.59] 0.44 [0.32, 0.54]
     4 PC1 198      0.80 [0.74, 0.85]      0.80 [0.74, 0.85]   0.0 0.80 [0.74, 0.85] 0.78 [0.72, 0.83]
     4 PC2 198      0.53 [0.42, 0.62]      0.53 [0.42, 0.62]   0.0 0.54 [0.43, 0.63] 0.49 [0.38, 0.59]
     5 PC1 198      0.80 [0.74, 0.85]      0

In [ ]:
summary.to_csv(RESULTS_DIR/"OOS_PC_estimation/oos_icc_odd_even_split_half_summary.csv")

full.to_csv(RESULTS_DIR/"OOS_PC_estimation/oos_icc_odd_even_split_half_full.csv")

In [ ]:
temporal_split_result = run_cross_fitted_analysis(
    data=data,
    split_mode="temporal",
    n_splits=N_FOLDS,
    n_jobs_outer=4,
    n_jobs_model=8,
    min_meals_per_half=MIN_MEALS_PER_HALF,
)

In [14]:
print("temporal split estimations :")
summary, full = icc_split_half(temporal_split_result["scores"])
print(summary.to_string(index=False))

temporal split estimations :
  fold  pc   n  ICC(A,1)  ICC(A,1) CI  ICC(C,1)  ICC(C,1) CI  pval    r         r CI  rho       rho CI
     1 PC1 199      0.73 [0.66, 0.79]      0.73 [0.66, 0.79]   0.0 0.73 [0.66, 0.79] 0.71 [0.63, 0.77]
     1 PC2 199      0.27 [0.14, 0.40]      0.28 [0.15, 0.40]   0.0 0.28 [0.15, 0.40] 0.31 [0.18, 0.43]
     2 PC1 199      0.72 [0.64, 0.78]      0.72 [0.64, 0.78]   0.0 0.72 [0.64, 0.78] 0.71 [0.64, 0.78]
     2 PC2 199      0.61 [0.51, 0.69]      0.61 [0.51, 0.69]   0.0 0.61 [0.51, 0.69] 0.48 [0.36, 0.58]
     3 PC1 198      0.78 [0.72, 0.83]      0.78 [0.72, 0.83]   0.0 0.79 [0.73, 0.84] 0.69 [0.61, 0.76]
     3 PC2 198      0.46 [0.34, 0.56]      0.46 [0.34, 0.56]   0.0 0.46 [0.35, 0.57] 0.41 [0.29, 0.52]
     4 PC1 198      0.73 [0.66, 0.79]      0.73 [0.65, 0.79]   0.0 0.73 [0.66, 0.79] 0.68 [0.60, 0.75]
     4 PC2 198      0.52 [0.41, 0.61]      0.52 [0.41, 0.61]   0.0 0.52 [0.41, 0.62] 0.50 [0.39, 0.60]
     5 PC1 198      0.73 [0.66, 0.79]      0

In [15]:
temporal_split_result["diagnostics"]

,fold,split_mode,n_train_subjects,n_test_subjects,n_scored_subjects,converged,log_likelihood,pc1_variance_fraction,pc2_variance_fraction
0,1,temporal,793,199,199,True,-88429.275972,0.665357,0.130563
1,2,temporal,793,199,199,True,-89583.285474,0.682171,0.120676
2,3,temporal,794,198,198,True,-89469.668162,0.668710,0.115054
3,4,temporal,794,198,198,True,-88724.091488,0.700232,0.110309
4,5,temporal,794,198,198,True,-90223.928526,0.672933,0.115917


In [16]:
summary.to_csv(RESULTS_DIR/"OOS_PC_estimation/oos_icc_temporal_split_half_summary.csv")

full.to_csv(RESULTS_DIR/"OOS_PC_estimation/oos_icc_temporal_split_half_full.csv")

## OOS three-day versus eleven-day analysis

The second cross-fitted analysis applies the same frozen-fold principle to three-day versus eleven-day phenotype estimates.

In [17]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed


def prepare_monitoring_horizon(
    data,
    subject_col=SUBJECT_COL,
    date_col=DATE_COL,
    monitor_day_col="eaten_date",
    horizon_days=14,
    eligible_subjects=None,
):
    """
    Add analysis-only monitoring-day fields and retain days 1..horizon_days.

    Prefer passing the actual 1-indexed study/monitoring-day column. If
    monitor_day_col is None, day 1 is inferred from the first logged calendar day.
    """
    out = data.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    if monitor_day_col is None:
        anchor = out.groupby(subject_col)[date_col].transform("min").dt.normalize()
        out["_analysis_start"] = anchor
        out["_analysis_day"] = (
            (out[date_col].dt.normalize() - anchor).dt.days + 1
        ).astype(int)
    else:
        out["_analysis_day"] = pd.to_numeric(
            out[monitor_day_col], errors="coerce"
        )
        out = out.dropna(subset=["_analysis_day"]).copy()
        out["_analysis_day"] = out["_analysis_day"].astype(int)

        implied_start = (
            out[date_col].dt.normalize()
            - pd.to_timedelta(out["_analysis_day"] - 1, unit="D")
        )
        out["_analysis_start"] = implied_start.groupby(
            out[subject_col]
        ).transform("min")

    out = out[out["_analysis_day"].between(1, horizon_days)].copy()

    # Prefer supplying the same predefined 14-day cohort used in the manuscript.
    if eligible_subjects is None:
        eligible_subjects = (
            out.groupby(subject_col)["_analysis_day"]
            .max()
            .loc[lambda x: x >= horizon_days]
            .index
        )

    return out[out[subject_col].isin(eligible_subjects)].copy()


def split_three_day_vs_complement(
    prepared_data,
    window_start_day,
    min_meals_3d,
    min_meals_11d,
    subject_col=SUBJECT_COL,
    date_col=DATE_COL,
    window_days=3,
    horizon_days=14,
    lookback_hours=6,
    response_hours=2,
):
    """
    Return one contiguous 3-day subset and its disjoint 11-day complement.

    Boundary rows are excluded when their 6-hour predictor history or 2-hour
    PPGR window would cross from one subset into the other.
    """
    window_end_day = window_start_day + window_days - 1

    if window_start_day < 1 or window_end_day > horizon_days:
        raise ValueError(
            "The requested window falls outside the monitoring horizon."
        )

    in_window = prepared_data["_analysis_day"].between(
        window_start_day, window_end_day
    )
    data_3d = prepared_data.loc[in_window].copy()
    data_11d = prepared_data.loc[~in_window].copy()

    lookback = pd.Timedelta(hours=lookback_hours)
    response = pd.Timedelta(hours=response_hours)

    # Beginning of an interior 3-day window:
    # - remove early 3-day meals whose history reaches into the complement;
    # - remove preceding complement meals whose PPGR reaches into the window.
    if window_start_day > 1:
        boundary_3d = (
            data_3d["_analysis_start"]
            + pd.to_timedelta(window_start_day - 1, unit="D")
        )
        data_3d = data_3d[
            data_3d[date_col] >= boundary_3d + lookback
        ]

        boundary_11d = (
            data_11d["_analysis_start"]
            + pd.to_timedelta(window_start_day - 1, unit="D")
        )
        crosses_start = (
            (data_11d[date_col] >= boundary_11d - response)
            & (data_11d[date_col] < boundary_11d)
        )
        data_11d = data_11d[~crosses_start]

    # End of an interior 3-day window:
    # - remove late 3-day meals whose PPGR reaches into the complement;
    # - remove following complement meals whose history reaches into the window.
    if window_end_day < horizon_days:
        boundary_3d = (
            data_3d["_analysis_start"]
            + pd.to_timedelta(window_end_day, unit="D")
        )
        data_3d = data_3d[
            data_3d[date_col] < boundary_3d - response
        ]

        boundary_11d = (
            data_11d["_analysis_start"]
            + pd.to_timedelta(window_end_day, unit="D")
        )
        crosses_end = (
            (data_11d[date_col] >= boundary_11d)
            & (data_11d[date_col] < boundary_11d + lookback)
        )
        data_11d = data_11d[~crosses_end]

    counts_3d = data_3d.groupby(subject_col).agg(
        n_meals_3d=(date_col, "size"),
        n_days_3d=("_analysis_day", "nunique"),
    )
    counts_11d = data_11d.groupby(subject_col).agg(
        n_meals_11d=(date_col, "size"),
        n_days_11d=("_analysis_day", "nunique"),
    )

    counts = counts_3d.join(counts_11d, how="inner")
    counts = counts[
        (counts["n_meals_3d"] >= min_meals_3d)
        & (counts["n_meals_11d"] >= min_meals_11d)
    ].copy()

    eligible = counts.index
    helper_cols = ["_analysis_start", "_analysis_day"]

    data_3d = (
        data_3d[data_3d[subject_col].isin(eligible)]
        .sort_values([subject_col, date_col])
        .drop(columns=helper_cols, errors="ignore")
    )
    data_11d = (
        data_11d[data_11d[subject_col].isin(eligible)]
        .sort_values([subject_col, date_col])
        .drop(columns=helper_cols, errors="ignore")
    )

    counts = counts.reset_index()
    counts["window_start_day"] = window_start_day
    counts["window_end_day"] = window_end_day

    return data_3d, data_11d, counts


def process_three_vs_eleven_fold(
    fold_id,
    train_subjects,
    test_subjects,
    data,
    min_meals_3d,
    min_meals_11d,
    window_starts=range(1, 13),
    horizon_days=14,
    window_days=3,
    monitor_day_col=None,
    eligible_subjects=None,
    lookback_hours=6,
    response_hours=2,
    n_jobs_model=4,
    subject_col=SUBJECT_COL,
    date_col=DATE_COL,
):
    """
    Fit one population fold and estimate paired 3-day/11-day PC scores in
    held-out participants under the same frozen model and PC basis.
    """
    train_data = data[data[subject_col].isin(train_subjects)].copy()
    test_data = data[data[subject_col].isin(test_subjects)].copy()

    if set(train_subjects).intersection(test_subjects):
        raise RuntimeError(
            f"Training/test participant leakage in fold {fold_id}."
        )

    prepared_test = prepare_monitoring_horizon(
        test_data,
        subject_col=subject_col,
        date_col=date_col,
        monitor_day_col=monitor_day_col,
        horizon_days=horizon_days,
        eligible_subjects=eligible_subjects,
    )

    if prepared_test.empty:
        raise ValueError(
            f"No eligible {horizon_days}-day participants in fold {fold_id}."
        )

    # Population surface, covariance, scaling and PC basis use training people only.
    fitted_result, fold_scalers, train_info = fit_training_fold(
        train_data,
        n_jobs_model=n_jobs_model,
    )
    eigenvalues, eigenvectors = derive_pc_basis(fitted_result)

    score_parts = []
    diagnostic_rows = []
    loading_table = None

    for start_day in window_starts:
        data_3d, data_11d, counts = split_three_day_vs_complement(
            prepared_test,
            window_start_day=start_day,
            min_meals_3d=min_meals_3d,
            min_meals_11d=min_meals_11d,
            subject_col=subject_col,
            date_col=date_col,
            window_days=window_days,
            horizon_days=horizon_days,
            lookback_hours=lookback_hours,
            response_hours=response_hours,
        )

        if counts.empty:
            continue

        # Only the held-out participant BLUP is estimated; MMER is not refitted.
        blup_3d = infer_blups_from_frozen_model(
            fitted_result,
            data_3d,
            fold_scalers,
        )
        blup_11d = infer_blups_from_frozen_model(
            fitted_result,
            data_11d,
            fold_scalers,
        )

        common = blup_3d.index.intersection(blup_11d.index)
        if len(common) < 3:
            continue

        blup_3d = blup_3d.loc[common]
        blup_11d = blup_11d.loc[common]

        if list(blup_3d.columns) != list(blup_11d.columns):
            raise RuntimeError(
                "BLUP coordinate order differs between 3-day and 11-day estimates."
            )

        scores_3d = project_blups_to_pcs(
            blup_3d,
            eigenvalues,
            eigenvectors,
            n_components=2,
            standardize=True,
        )
        scores_11d = project_blups_to_pcs(
            blup_11d,
            eigenvalues,
            eigenvectors,
            n_components=2,
            standardize=True,
        )

        scores = (
            scores_3d.add_suffix("_3d")
            .join(scores_11d.add_suffix("_11d"))
            .reset_index()
        )

        index_name = blup_3d.index.name or "index"
        scores = scores.rename(columns={index_name: subject_col})
        scores["fold"] = fold_id
        scores["window_start_day"] = start_day
        scores["window_end_day"] = start_day + window_days - 1

        scores = scores.merge(
            counts,
            on=[
                subject_col,
                "window_start_day",
                "window_end_day",
            ],
            how="left",
        )
        score_parts.append(scores)

        diagnostic_rows.append({
            "fold": fold_id,
            "window_start_day": start_day,
            "window_end_day": start_day + window_days - 1,
            "n_train_subjects": len(train_subjects),
            "n_test_subjects": len(test_subjects),
            "n_14d_test_subjects": prepared_test[subject_col].nunique(),
            "n_scored_subjects": len(scores),
            "n_train_meals": len(train_data),
            "converged": fitted_result.is_converged,
            "log_likelihood": fitted_result.best_log_likelihood,
            "pc1_variance_fraction": eigenvalues[0] / eigenvalues.sum(),
            "pc2_variance_fraction": eigenvalues[1] / eigenvalues.sum(),
        })

        if loading_table is None:
            loading_table = pd.DataFrame({
                "coordinate": blup_3d.columns,
                "PC1": eigenvectors[:, 0],
                "PC2": eigenvectors[:, 1],
                "fold": fold_id,
            })

    if not score_parts:
        raise ValueError(
            f"No participants passed the window criteria in fold {fold_id}."
        )

    return {
        "scores": pd.concat(score_parts, ignore_index=True),
        "diagnostics": pd.DataFrame(diagnostic_rows),
        "loadings": loading_table,
        "train_info": train_info,
    }


def run_cross_fitted_three_vs_eleven(
    data,
    min_meals_3d=5,
    min_meals_11d=10,
    window_starts=range(1, 13),
    n_splits=5,
    n_jobs_outer=2,
    n_jobs_model=4,
    horizon_days=14,
    window_days=3,
    monitor_day_col=None,
    eligible_subjects=None,
    lookback_hours=6,
    response_hours=2,
    subject_col=SUBJECT_COL,
    date_col=DATE_COL,
):
    """Run participant-level cross-fitting for the requested 3-day windows."""
    window_starts = tuple(window_starts)
    folds = make_subject_folds(data, n_splits=n_splits)

    results = Parallel(
        n_jobs=n_jobs_outer,
        backend="loky",
        verbose=10,
    )(
        delayed(process_three_vs_eleven_fold)(
            fold_id=fold["fold_id"],
            train_subjects=fold["train_subjects"],
            test_subjects=fold["test_subjects"],
            data=data,
            min_meals_3d=min_meals_3d,
            min_meals_11d=min_meals_11d,
            window_starts=window_starts,
            horizon_days=horizon_days,
            window_days=window_days,
            monitor_day_col=monitor_day_col,
            eligible_subjects=eligible_subjects,
            lookback_hours=lookback_hours,
            response_hours=response_hours,
            n_jobs_model=n_jobs_model,
            subject_col=subject_col,
            date_col=date_col,
        )
        for fold in folds
    )

    return {
        "scores": pd.concat(
            [result["scores"] for result in results],
            ignore_index=True,
        ),
        "diagnostics": pd.concat(
            [result["diagnostics"] for result in results],
            ignore_index=True,
        ),
        "loadings": pd.concat(
            [result["loadings"] for result in results],
            ignore_index=True,
        ),
        "train_info": [
            result["train_info"]
            for result in results
        ],
    }

In [18]:
#### post-hoc analyses
def paired_icc(x, y):
    """Two-way absolute-agreement ICC(2,1) and consistency ICC(3,1)."""
    values = np.column_stack([
        np.asarray(x, dtype=float),
        np.asarray(y, dtype=float),
    ])
    values = values[np.isfinite(values).all(axis=1)]
    n, k = values.shape

    if n < 3:
        return np.nan, np.nan

    grand_mean = values.mean()
    subject_means = values.mean(axis=1)
    measurement_means = values.mean(axis=0)

    ms_subject = (
        k * np.square(subject_means - grand_mean).sum() / (n - 1)
    )
    ms_measurement = (
        n * np.square(measurement_means - grand_mean).sum() / (k - 1)
    )

    residual = (
        values
        - subject_means[:, None]
        - measurement_means[None, :]
        + grand_mean
    )
    ms_error = (
        np.square(residual).sum()
        / ((n - 1) * (k - 1))
    )

    denominator_absolute = (
        ms_subject
        + (k - 1) * ms_error
        + k * (ms_measurement - ms_error) / n
    )
    denominator_consistency = (
        ms_subject
        + (k - 1) * ms_error
    )

    icc_absolute = (
        (ms_subject - ms_error) / denominator_absolute
        if denominator_absolute != 0
        else np.nan
    )
    icc_consistency = (
        (ms_subject - ms_error) / denominator_consistency
        if denominator_consistency != 0
        else np.nan
    )

    return icc_absolute, icc_consistency


def agreement_stats(frame, short_col, reference_col):
    """Agreement and shrinkage diagnostics for paired 3-day/11-day scores."""
    paired = frame[[short_col, reference_col]].dropna()
    x = paired[short_col].to_numpy(dtype=float)
    y = paired[reference_col].to_numpy(dtype=float)
    n = len(paired)

    empty = {
        "n": n,
        "icc2_1": np.nan,
        "icc3_1": np.nan,
        "pearson_r": np.nan,
        "spearman_r": np.nan,
        "bias_3d_minus_11d": np.nan,
        "mae": np.nan,
        "rmse": np.nan,
        "sd_ratio_3d_to_11d": np.nan,
        "calibration_slope": np.nan,
        "calibration_intercept": np.nan,
    }
    if n < 3:
        return empty

    icc_absolute, icc_consistency = paired_icc(x, y)
    sd_x = x.std(ddof=1)
    sd_y = y.std(ddof=1)
    var_y = y.var(ddof=1)

    slope = (
        np.cov(y, x, ddof=1)[0, 1] / var_y
        if var_y > 0
        else np.nan
    )
    intercept = (
        x.mean() - slope * y.mean()
        if np.isfinite(slope)
        else np.nan
    )
    difference = x - y

    return {
        "n": n,
        "icc2_1": icc_absolute,
        "icc3_1": icc_consistency,
        "pearson_r": (
            np.corrcoef(x, y)[0, 1]
            if sd_x > 0 and sd_y > 0
            else np.nan
        ),
        "spearman_r": pd.Series(x).corr(
            pd.Series(y), method="spearman"
        ),
        "bias_3d_minus_11d": difference.mean(),
        "mae": np.abs(difference).mean(),
        "rmse": np.sqrt(np.square(difference).mean()),
        "sd_ratio_3d_to_11d": (
            sd_x / sd_y
            if sd_y > 0
            else np.nan
        ),
        "calibration_slope": slope,
        "calibration_intercept": intercept,
    }


def bootstrap_agreement_ci(
    frame,
    short_col,
    reference_col,
    n_boot=1000,
    random_state=42,
    strata_col="fold",
):
    """
    Participant bootstrap, stratified by population fold when available.

    These intervals are conditional on the five fitted population models;
    the MMER models are not refitted within every bootstrap iteration.
    """
    columns = [short_col, reference_col]
    if strata_col in frame.columns:
        columns.append(strata_col)

    paired = frame[columns].dropna()
    if len(paired) < 3 or n_boot <= 0:
        return {}

    if strata_col in paired.columns:
        arrays = [
            group[[short_col, reference_col]].to_numpy(dtype=float)
            for _, group in paired.groupby(strata_col, sort=False)
        ]
    else:
        arrays = [
            paired[[short_col, reference_col]].to_numpy(dtype=float)
        ]

    rng = np.random.default_rng(random_state)
    bootstrap_values = {
        "icc2_1": [],
        "icc3_1": [],
        "calibration_slope": [],
        "sd_ratio_3d_to_11d": [],
    }

    for _ in range(n_boot):
        sample = np.vstack([
            values[
                rng.integers(0, len(values), size=len(values))
            ]
            for values in arrays
        ])
        sample = pd.DataFrame(
            sample,
            columns=[short_col, reference_col],
        )
        stats = agreement_stats(
            sample,
            short_col,
            reference_col,
        )

        for metric in bootstrap_values:
            bootstrap_values[metric].append(stats[metric])

    intervals = {}
    for metric, values in bootstrap_values.items():
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]

        if len(values):
            lower, upper = np.percentile(values, [2.5, 97.5])
        else:
            lower, upper = np.nan, np.nan

        intervals[f"{metric}_ci_low"] = lower
        intervals[f"{metric}_ci_high"] = upper

    return intervals


def restrict_to_complete_window_panel(
    scores,
    subject_col=SUBJECT_COL,
):
    """Retain participants who were scored for every requested window."""
    n_windows = scores["window_start_day"].nunique()
    participant_n_windows = (
        scores.groupby(subject_col)["window_start_day"].nunique()
    )
    complete_subjects = participant_n_windows[
        participant_n_windows == n_windows
    ].index

    return scores[
        scores[subject_col].isin(complete_subjects)
    ].copy()


def summarize_three_vs_eleven(
    scores,
    pcs=("PC1", "PC2"),
    n_boot=1000,
    random_state=42,
    subject_col=SUBJECT_COL,
):
    """
    Return window-specific pooled estimates, fold-specific checks, and
    median/range summaries across window positions.
    """
    duplicate_keys = [
        subject_col,
        "window_start_day",
        "window_end_day",
    ]
    if scores.duplicated(duplicate_keys).any():
        raise ValueError(
            "A participant has multiple rows for the same 3-day window."
        )

    by_window_rows = []
    seed = random_state

    for (start_day, end_day), frame in tqdm(scores.groupby(
        ["window_start_day", "window_end_day"],
        sort=True,
    )):
        for pc in pcs:
            short_col = f"{pc}_3d"
            reference_col = f"{pc}_11d"

            stats = agreement_stats(
                frame,
                short_col,
                reference_col,
            )
            intervals = bootstrap_agreement_ci(
                frame,
                short_col,
                reference_col,
                n_boot=n_boot,
                random_state=seed,
                strata_col="fold",
            )
            seed += 1

            by_window_rows.append({
                "pc": pc,
                "window_start_day": start_day,
                "window_end_day": end_day,
                **stats,
                **intervals,
            })

    by_fold_rows = []
    for (fold, start_day, end_day), frame in scores.groupby(
        ["fold", "window_start_day", "window_end_day"],
        sort=True,
    ):
        for pc in pcs:
            stats = agreement_stats(
                frame,
                f"{pc}_3d",
                f"{pc}_11d",
            )
            by_fold_rows.append({
                "fold": fold,
                "pc": pc,
                "window_start_day": start_day,
                "window_end_day": end_day,
                **stats,
            })

    by_window = pd.DataFrame(by_window_rows)
    by_fold = pd.DataFrame(by_fold_rows)

    overall_rows = []
    for pc, frame in by_window.groupby("pc", sort=False):
        overall_rows.append({
            "pc": pc,
            "n_windows": frame["window_start_day"].nunique(),
            "median_n": frame["n"].median(),
            "median_icc2_1": frame["icc2_1"].median(),
            "min_icc2_1": frame["icc2_1"].min(),
            "max_icc2_1": frame["icc2_1"].max(),
            "median_icc3_1": frame["icc3_1"].median(),
            "median_pearson_r": frame["pearson_r"].median(),
            "median_sd_ratio_3d_to_11d": (
                frame["sd_ratio_3d_to_11d"].median()
            ),
            "median_calibration_slope": (
                frame["calibration_slope"].median()
            ),
            "windows_icc2_ge_0_75": int(
                (frame["icc2_1"] >= 0.75).sum()
            ),
        })

    return {
        "by_window": (
            by_window
            .sort_values(["pc", "window_start_day"])
            .reset_index(drop=True)
        ),
        "by_fold": (
            by_fold
            .sort_values(["pc", "window_start_day", "fold"])
            .reset_index(drop=True)
        ),
        "overall": pd.DataFrame(overall_rows),
        "primary_days_1_3": (
            by_window[
                by_window["window_start_day"] == 1
            ]
            .reset_index(drop=True)
        ),
    }

In [ ]:
three_vs_eleven_result = run_cross_fitted_three_vs_eleven(
    data=data[data["is_standardized_meal"]==False].reset_index().copy(),
    window_starts=range(1, 13),
    n_splits=N_FOLDS,
    n_jobs_outer=5,
    n_jobs_model=6,
    horizon_days=14,
    window_days=3,
    lookback_hours=6,
    response_hours=2,
)

[Parallel(n_jobs=5)]: Using backend LokyBackend with 5 concurrent workers.
Finished: no further improvement!:  32%|███▏      | 16/50 06:04         
Finished: no further improvement!:  40%|████      | 20/50 07:36         
Finished: no further improvement!:  42%|████▏     | 21/50 07:37         
Finished: no further improvement!:  50%|█████     | 25/50 08:58         
Finished: no further improvement!:  50%|█████     | 25/50 08:56         
Process LokyProcess-17:
Process LokyProcess-19:
Traceback (most recent call last):
Traceback (most recent call last):
  File "lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "site-packages/joblib/externals/loky/process_executor.py", line 476, in _process_worker
    result_queue.put(pid)
  File "site-packages/joblib/externals/loky/backend/queues.py", line 235, in put
    with self._wlock:
  File "site

In [ ]:
three_vs_eleven_scores = three_vs_eleven_result["scores"]

three_vs_eleven_posthoc = summarize_three_vs_eleven(
    three_vs_eleven_scores,
    pcs=("PC1", "PC2"),
    n_boot=1000,
    random_state=42,
)

primary_result = three_vs_eleven_posthoc["primary_days_1_3"]
window_results = three_vs_eleven_posthoc["by_window"]
fold_results = three_vs_eleven_posthoc["by_fold"]
overall_result = three_vs_eleven_posthoc["overall"]

100%|██████████| 12/12 [00:37<00:00,  3.11s/it]


In [ ]:
primary_result

,pc,window_start_day,window_end_day,n,icc2_1,icc3_1,pearson_r,spearman_r,bias_3d_minus_11d,mae,...,calibration_slope,calibration_intercept,icc2_1_ci_low,icc2_1_ci_high,icc3_1_ci_low,icc3_1_ci_high,calibration_slope_ci_low,calibration_slope_ci_high,sd_ratio_3d_to_11d_ci_low,sd_ratio_3d_to_11d_ci_high
0,PC1,1,3,191,0.559456,0.565599,0.565870,0.581055,-0.138242,0.608547,...,0.583651,-0.225729,0.431418,0.665807,0.442114,0.672219,0.438507,0.752123,0.916604,1.178254
1,PC2,1,3,191,0.386889,0.385663,0.388674,0.351150,-0.006855,0.583521,...,0.343049,0.046857,0.253843,0.505898,0.253868,0.505140,0.218404,0.482176,0.744629,1.030381


In [ ]:
primary_result.round(2)

,pc,window_start_day,window_end_day,n,icc2_1,icc3_1,pearson_r,spearman_r,bias_3d_minus_11d,mae,...,calibration_slope,calibration_intercept,icc2_1_ci_low,icc2_1_ci_high,icc3_1_ci_low,icc3_1_ci_high,calibration_slope_ci_low,calibration_slope_ci_high,sd_ratio_3d_to_11d_ci_low,sd_ratio_3d_to_11d_ci_high
0,PC1,1,3,818,0.655155,0.665273,0.665291,0.626621,-0.178204,0.644061,...,0.670256,-0.133380,0.602678,0.699605,0.612850,0.712023,0.601541,0.738305,0.941050,1.078229
1,PC2,1,3,818,0.448078,0.449041,0.457442,0.355889,0.060454,0.657764,...,0.377100,0.051888,0.347876,0.543480,0.349152,0.546369,0.288453,0.471540,0.769279,0.886381


In [ ]:
d = three_vs_eleven_result["diagnostics"]

d1 = d[d["window_start_day"] == 1].copy()

display(
    d1[
        [
            "fold",
            "n_test_subjects",
            "n_14d_test_subjects",
            "n_scored_subjects",
        ]
    ]
)

print(
    d1[
        [
            "n_test_subjects",
            "n_14d_test_subjects",
            "n_scored_subjects",
        ]
    ].sum()
)

,fold,n_test_subjects,n_14d_test_subjects,n_scored_subjects
0,1,199,163,162
12,2,199,168,167
24,3,198,171,167
36,4,198,174,170
48,5,198,157,152


n_test_subjects        992
n_14d_test_subjects    833
n_scored_subjects      818
dtype: int64


In [ ]:
s = three_vs_eleven_scores[
    three_vs_eleven_scores["window_start_day"] == 1
]

display(
    s[["n_meals_3d", "n_meals_11d"]]
    .describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
)

,n_meals_3d,n_meals_11d
count,818.000000,818.000000
mean,10.660147,35.432763
std,1.769980,5.641558
min,5.000000,10.000000
5%,8.000000,25.000000
10%,8.000000,28.000000
25%,10.000000,32.000000
50%,11.000000,36.000000
75%,12.000000,39.000000
90%,13.000000,42.000000


In [ ]:
display(
    window_results.loc[
        window_results["pc"] == "PC1",
        [
            "window_start_day",
            "window_end_day",
            "n",
            "icc2_1",
            "icc2_1_ci_low",
            "icc2_1_ci_high",
            "pearson_r",
            "bias_3d_minus_11d",
            "sd_ratio_3d_to_11d",
            "calibration_slope",
        ],
    ]
)

,window_start_day,window_end_day,n,icc2_1,icc2_1_ci_low,icc2_1_ci_high,pearson_r,bias_3d_minus_11d,sd_ratio_3d_to_11d,calibration_slope
0,1,3,818,0.655155,0.602678,0.699605,0.665291,-0.178204,1.007463,0.670256
1,2,4,802,0.697010,0.644384,0.740629,0.698153,-0.061483,0.995908,0.695296
2,3,5,796,0.668597,0.605044,0.721839,0.669191,0.000661,1.052413,0.704265
3,4,6,793,0.666619,0.607154,0.718018,0.666515,0.001571,1.023198,0.681977
4,5,7,790,0.658370,0.596748,0.712693,0.658394,0.012344,0.972490,0.640282
5,6,8,785,0.690949,0.634243,0.740226,0.692175,0.051250,0.960502,0.664835
6,7,9,769,0.736002,0.690040,0.777922,0.737027,0.036493,0.955379,0.704140
7,8,10,762,0.719334,0.671152,0.764925,0.719688,0.002573,0.959428,0.690490
8,9,11,756,0.695377,0.645190,0.740864,0.697178,0.005584,0.925753,0.645414
9,10,12,741,0.659337,0.602688,0.709157,0.663805,-0.018366,0.888014,0.589468


In [ ]:
display(
    fold_results.loc[
        (fold_results["pc"] == "PC1")
        & (fold_results["window_start_day"] == 1),
        [
            "fold",
            "n",
            "icc2_1",
            "icc3_1",
            "pearson_r",
            "bias_3d_minus_11d",
        ],
    ]
)

,fold,n,icc2_1,icc3_1,pearson_r,bias_3d_minus_11d
0,1,162,0.646512,0.661644,0.661658,-0.210998
1,2,167,0.667734,0.684281,0.685187,-0.229202
2,3,167,0.637478,0.639855,0.644458,-0.117828
3,4,170,0.663208,0.668834,0.672694,-0.137625
4,5,152,0.663211,0.674062,0.674339,-0.198941
